# Size does not matter: Sub-Billion VLM NanoChimera is All You Need!

This project addresses the modern challenge of Vision-Language Model (VLM) deployment by testing the scalability hypothesis: **Is massive parameter count necessary for effective visual reasoning?**

We construct and train **NanoChimera VLM**, a novel modular architecture designed to achieve high performance while strictly remaining **Sub-Billion** parameters ($\approx 999.99 \text{ Million}$ total deployed). Our goal is threefold: to provide a practical tutorial on building custom VLM architectures by training the critical Projector Layer, to rigorously evaluate its cognitive capabilities (grounding and hallucination), and to serve as a proof-of-concept for **real-time, edge-friendly multimodal AI**.


First of all, lets import the libraries and set a common seed 42 so the experiment is reproducible.


The outline of the notebook goes as following:

0) Data Loading and Augmentations
1) Model Architecture
2) Training Pipeline
3) Evaluation Pipeline
4) Experiments
5) Results
6) Conclusions


---

1. Add flags (do visft...)
2. Think of whichs experiments to perform
3.

In [ ]:
# WARNING: THE DAMN COLLAB ENVIRONMENTS VERSIONS OF THE LIBRARIES ARE TRASH RELOAD THE KERNEL UNTIL NO NUMPY INCOMPATIBILITY IS FOUND :)
# TODO: FIX THIS FUCKING SLOP!!!!!!
######## EXECUTION CONFIGURATION FLAGS
RUNNING_ON_GOOGLE = True
PERFORM_EVALUATION =  True
PERFORM_TRAINING = False
USE_SMALL_DATA = True

config = {}

# Regular python related
import os
import time
import pickle
import random
import math
import json
from datetime import datetime
from typing import *

if RUNNING_ON_GOOGLE:
  from google.colab import drive, userdata
  os.environ["HUGGINGFACE_TOKEN"] = userdata.get("HUGGINGFACE_TOKEN")
  os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
  os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

  # drive.mount('/content/drive')

  # %pip install numpy==2.0
  %pip install loguru
  %pip install lmms-eval


import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from loguru import logger
from tqdm import tqdm


# Pytorch related
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split

from lmms_eval.api.instance import Instance
from lmms_eval.models import get_model
from lmms_eval.tasks import TaskManager, get_task_dict
from lmms_eval.api.model import lmms
from lmms_eval.evaluator import simple_evaluate
# HuggingFace related
from datasets import load_dataset
import huggingface_hub
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, AutoModel

def set_seed(seed, use_gpu = True, change_numpy_seed = False):
    random.seed(seed)
    # This may cause problems with CUDA
    if change_numpy_seed:
      np.random.seed(seed)
    torch.manual_seed(seed)
    if use_gpu:
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

SEED = 42
USE_SEED = True
if USE_SEED:
    set_seed(SEED, torch.cuda.is_available())

huggingface_hub.login(token=os.environ.get("HUGGINGFACE_TOKEN"))

## 0. Data Loading & Augmentations

Thanks to the already well curated efforts by meta with the LLaVa model, we do not need to preprocess the data at all, its just a plug and play clean dataset. Now it has to be said that if we wanted to artificially augment the size of the dataset this could easily be done throught common computer vision data augmentation techniques, however due to the sheer data volume we dispose. Nevertheless, for completeness, below we add a set of transformations that could be easily plugged into the Torch dataset API to perform data augmentation.

In [ ]:
class VLM_Dataset(Dataset):
    def __init__(
        self,
        size=10000,
        filename="training_dataset.pkl",
        dataset="damerajee/Llava-pretrain-small",
        split="train",
        overwrite=False,
        seed=42,
        transform=None
    ):
        self.transform = transform

        # Load from pickle if available
        if os.path.exists(filename) and not overwrite:
            with open(filename, "rb") as f:
                self.data = pickle.load(f)
            return

        # Otherwise, read dataset from HuggingFace
        stream = load_dataset(
            dataset,
            split=split,
            streaming=True
        ).shuffle(buffer_size=size, seed=seed)

        self.data = []
        for i, row in enumerate(stream):
            if i >= size:
                break

            # Prepend <image> token to caption
            caption_with_image = "<image> " + row["answer"]

            self.data.append({
                "image": row["image"],
                "caption": caption_with_image
            })

        # Save dataset for future use
        with open(filename, "wb") as f:
            pickle.dump(self.data, f)

    # ---------- PyTorch API ----------
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        if self.transform:
            item["image"] = self.transform(item["image"])
        return item

In [ ]:
# Data augmentation transformations
"""
transforms = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224),
    torchvision.transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    torchvision.transforms.RandomHorizontalFlip(p=0.3),
    torchvision.transforms.RandomVerticalFlip(p=0.3),
    torchvision.transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2,hue=0.05),
    torchvision.transforms.RandomGrayscale(p=0.1),
    torchvision.transforms.RandomAdjustSharpness(1.5, p=0.2),
    torchvision.transforms.ToTensor(),
])

# Basic transformation
transforms = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224),
    torchvision.transforms.ToTensor(),
])
"""

In [ ]:
# Example of applying transformations to K images
"""
dataset = VLM_Dataset(
    size=50,
    transform=transforms
)
loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)


# Get a batch
batch = next(iter(loader))
images = batch["image"]
captions = batch["caption"]
K = images.size(0)
cols = 4
rows = math.ceil(K / cols)

plt.figure(figsize=(cols * 4, rows * 4))

for i in range(K):
    img = images[i].permute(1, 2, 0)

    # If normalized, undo it (adjust if you used different stats)
    img = img * 0.5 + 0.5
    img = img.clamp(0, 1)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(img)
    plt.title(captions[i][:40], fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.show()

"""
None

## 1. Model Architecture

This is one of the single most important points of this project, the architecture of NanoChimera that will allow us to connect visual to textual tokens and thus allow the LLM to understand images. Therefore the visual adapter and its quality is the biggest factor determining the quality of this VLM.

The easiest implementation of this adapter is an MLP, and we can also relax the implicit assumption that all the image vision tokens should be used through the connector and passed down to the LLM, which is often not the case, some visual tokens have undoubtedly more importance than others.

In [ ]:
class VisionConnector(nn.Module):
    """
    Maps vision encoder features -> LLM embedding space
    Supports a flexible number of hidden layers.
    """
    def __init__(self, vision_dim, llm_dim, hidden_dims=(4096,), device = None):
        super().__init__()
        self.device = device

        layers = []
        # Start with vision_dim
        current_dim = vision_dim

        # Add hidden layers dynamically
        for h_dim in hidden_dims:
            layers.append(nn.Linear(current_dim, h_dim))
            layers.append(nn.GELU())
            current_dim = h_dim

        # Add the final projection layer to llm_dim
        layers.append(nn.Linear(current_dim, llm_dim))

        self.proj = nn.Sequential(*layers)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, vision_feats):
        # vision_feats: (B, N, vision_dim)
        return self.proj(vision_feats)  # (B, N, llm_dim)

In [ ]:
def merge_text_and_visual_embeddings(
    images,
    K,
    device,
    llm,
    tokenizer,
    connector,
    texts=None
):
    """
    Merge tokenized text embeddings (optional) with visual embeddings projected via VisionConnector.

    Args:
        images (Tensor): batch of images [B, C, H, W].
        K (int): number of visual tokens per image.
        device: torch device.
        llm: language model to get token embeddings.
        tokenizer: tokenizer used for text.
        connector: VisionConnector instance.
        texts (list[str] | None): optional list of captions or question+caption.
        IMAGE_TOKEN: string for image placeholder token.
    
    Returns:
        inputs_embeds: [B, L, llm_dim]
        attention_mask: [B, L]
        image_token_idx: list of indices of <image> tokens in each text
    """
    B = images.size(0)

    # 1. Default text if none provided
    if texts is None:
        texts = ["<image> Please describe the contents of this image in one sentence."] * B

    # --------------------------------------------------
    # 2. Tokenize
    # --------------------------------------------------
    encoded = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    input_ids = encoded.input_ids.to(device)
    attention_mask = encoded.attention_mask.to(device)
    text_embeds = llm.get_input_embeddings()(input_ids)  # [B, L, llm_dim]

    # --------------------------------------------------
    # 3. Encode images
    # --------------------------------------------------
    with torch.no_grad():
        vision_out = vision_model.vision_model(pixel_values=images.to(device))
        vision_feats = vision_out.last_hidden_state  # (B, N, vision_dim)

    vis_embeds = connector(vision_feats)[:, :K, :]  # (B, K, D)

    # --------------------------------------------------
    # 4. Merge embeddings
    # --------------------------------------------------
    image_token_id = tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)
    merged_embeds_list = []
    merged_masks_list = []
    image_token_idx = []
    max_len = 0

    for b in range(B):
        idxs = (input_ids[b] == image_token_id).nonzero(as_tuple=False)
        if len(idxs) == 0:
            raise ValueError(f"No <image> token found in text: {final_texts[b]}")
        image_token_idx.append(idxs[0].item())
        idx = image_token_idx[-1]

        merged = torch.cat([
            text_embeds[b, :idx],  # before <image>
            vis_embeds[b],         # visual tokens
            text_embeds[b, idx+1:] # after <image>
        ], dim=0)

        max_len = max(max_len, merged.size(0))
        merged_embeds_list.append(merged)

        # Attention mask (1 for real tokens, 0 for padding)
        pad_len = max_len - merged.size(0)
        merged_masks_list.append(torch.cat([
            torch.ones(merged.size(0) - pad_len, device=device),
            torch.zeros(pad_len, device=device)
        ]))

    inputs_embeds = torch.stack(merged_embeds_list, dim=0)
    attention_mask = torch.stack(merged_masks_list, dim=0)

    return inputs_embeds, attention_mask, image_token_idx

In [ ]:
def merge_text_and_visual_embeddings(
    images,
    K,
    device,
    llm,
    tokenizer,
    connector,
    texts=None
):
    B = images.size(0)

    if texts is None:
        texts = ["<image> Describe this image in one sentence."] * B

    # 1. Tokenize
    encoded = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    input_ids = encoded.input_ids.to(device)          # [B, L]
    attention_mask_text = encoded.attention_mask.to(device)
    text_embeds = llm.get_input_embeddings()(input_ids)  # [B, L, D]

    # 2. Encode images
    with torch.no_grad():
        vision_feats = vision_model.vision_model(pixel_values=images.to(device)).last_hidden_state
    vis_embeds = connector(vision_feats)[:, :K, :]  # [B, K, D]

    # 3. Find <image> token positions
    image_token_id = tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)
    image_token_idx = (input_ids == image_token_id).nonzero(as_tuple=True)[1]  # [B]

    # 4. Compute final sequence length
    L_text = input_ids.size(1)
    L_final = L_text + K - 1  # remove 1 token where <image> was
    D = text_embeds.size(2)

    # 5. Preallocate merged tensors
    inputs_embeds = torch.zeros(B, L_final, D, device=device)
    attention_mask = torch.zeros(B, L_final, device=device)

    # 6. Fill merged embeddings in batch
    for b in range(B):
        idx = image_token_idx[b].item()
        inputs_embeds[b, :idx] = text_embeds[b, :idx]
        inputs_embeds[b, idx:idx+K] = vis_embeds[b]
        inputs_embeds[b, idx+K:] = text_embeds[b, idx+1:]

        attention_mask[b, :idx+K] = 1
        attention_mask[b, idx+K:] = 1

    return inputs_embeds, attention_mask, image_token_idx.tolist()

In [ ]:
class NanoChimera(nn.Module):
    """
    LLaVA-style VLM:
    - Frozen vision encoder
    - Frozen LLM
    - Trainable connector
    """

    def __init__(self, vision_encoder, llm, connector):
        super().__init__()
        self.vision = vision_encoder
        self.llm = llm
        self.connector = connector

        self.vision.requires_grad_(False)
        self.llm.requires_grad_(False)

    def forward(
        self,
        images,
        input_ids,
        attention_mask,
        image_token_id,
        labels=None,
    ):
        # ---- vision ----
        with torch.no_grad():
            vision_out = self.vision(images)
            vision_feats = vision_out.last_hidden_state  # (B, N, vision_dim)

        # ---- connector ----
        visual_tokens = self.connector(vision_feats)  # (B, N, llm_dim)

        # ---- build inputs ----
        if labels is not None:
            inputs_embeds, attn_mask, labels = build_inputs(
                llm=self.llm,
                input_ids=input_ids,
                attention_mask=attention_mask,
                visual_tokens=visual_tokens,
                image_token_id=image_token_id,
                labels=labels,
            )
        else:
            inputs_embeds, attn_mask = build_inputs(
                llm=self.llm,
                input_ids=input_ids,
                attention_mask=attention_mask,
                visual_tokens=visual_tokens,
                image_token_id=image_token_id,
            )

        # ---- LLM ----
        return self.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=attn_mask,
            labels=labels,
        )

## 2. Training Pipeline definition

Now we will define a simple data splitting (train/validation/test) strategy along the training loop, to add some interesting MLOps features that will improve the quality of the training experience, we will add logging, persistent information of runs and managing checkpoints to always save the best models comparing to past models.

So the section is split into the following subsections:

1) Data splitting
2) Model setup
3) Discussion on simple training evaluation metrics
4) Training loop definition
5) Training example (**Pretraining**)
6) Pipeline extension (**Supervised Fine-Tuning**)




Below we can find a simple example of a pipeline hyperparameters configuration

In [ ]:
# This is a sample
EX_CONFIG = {
  # Data
  "dataset_size": 1024,
  "batch_size": 64,
  #"batch_size": 256,

  # Optimization algorithm
  "learning_rate": 9e-4,
  "weight_decay": 0.01,

  # Connector parameters
  "n_visual_tokens": 32,
  "connector_hidden_dims": (4096, ),

  # training
  "epochs": 2,
  "grad_accum_steps": 8,
  "warmup_ratio": 0.05,
  "max_grad_norm": 1.0,
  "scheduler_start_factor": 0.1,

  # logging
  "log_every": 32,
  "eval_every": 256,

  # criterion
  "label_smoothing": 0.05,
}

### 2.1 Splitting Strategy and Data Preparation

For now we won't make use of automatic cross validation and rely on a simple train/test/validation split. Thanks to Pytorch data loaders we can shuffle when sampling to avoid overfitting weird patterns.


In [ ]:
def split_dataset(dataset, train=0.8, val=0.1, test=0.1, seed=SEED):
    # Check for correct splitting
    assert train + val + test == 1.0
    # Getting lengths of datasets
    n = len(dataset)
    n_train = int(train * n)
    n_val = int(val * n)
    n_test = n - n_train - n_val
    # Using seed
    generator = torch.Generator().manual_seed(seed)

    return random_split(
        dataset,
        [n_train, n_val, n_test],
        generator=generator
    )

# For Pytorch Dataloader API
def collate_fn(batch, img_size=(224, 224)):
    # Keep PIL images for the processor and unifying pictures into one format
    img_transform = torchvision.transforms.Compose([
        torchvision.transforms.Resize(img_size),
        torchvision.transforms.Lambda(lambda img: img.convert("RGB")),  # force 3 channels
        torchvision.transforms.ToTensor()
    ])

    images = torch.stack([img_transform(item["image"]) for item in batch])
    captions = [item["caption"] for item in batch]
    return {"image": images, "caption": captions}

# LOAD DATA
data = VLM_Dataset(size=EX_CONFIG["dataset_size"], overwrite=True)
train_dataset, val_dataset, test_dataset = split_dataset(data)

train_loader = DataLoader(
    train_dataset, batch_size=EX_CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset, batch_size=EX_CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
)
test_loader = DataLoader(
    test_dataset, batch_size=EX_CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
)

### 2.2 Model setup

In [ ]:
# IMAGE to VISUAL TOKEN
def encode_image(imgs, device, K=32):
    """
    imgs: batch of PIL images or tensors
    K: number of visual tokens to keep
    """
    vision_model.eval()
    with torch.no_grad():
        # Vision processor handles batching automatically
        inputs = {"pixel_values": imgs.to(device)}
        # SigLIP / Vision model forward
        vision_out = vision_model.vision_model(**inputs)
        # Last hidden state: (B, N, vision_dim)
        feats = vision_out.last_hidden_state
    return feats

# BUILD MULTIMODAL EMBEDDINGS (TEXTUAL)
def build_inputs(batch, llm, tokenizer, EX_CONFIG, device):
    """
    Convert a batch from DataLoader into model-ready inputs with merged text + visual embeddings.
    """
    # 1. Prepare text (concatenate question and caption, no need to add <image> token again)
    questions = batch["question"]  # Add question from the batch
    captions = batch["caption"]
    
    # Concatenate question and caption (no <image> token needed here)
    texts = [f"{q}" for q in questions]  # Only the question, which already has the <image> token

    input_ids = tokenizer(
        texts, 
        return_tensors="pt", 
        padding=True, 
        truncation=True
    ).input_ids.to(device)

    attention_mask = torch.ones_like(input_ids, device=device)

    # 2. Encode images
    images = batch["image"].to(device)
    vis_embeds = encode_image(images, device=device, K=EX_CONFIG["n_visual_tokens"])

    # 3. Merge embeddings & adjust labels using shared Core
    inputs_embeds, attention_mask, image_token_idx = merge_text_and_visual_embeddings(
        images=images,
        K=EX_CONFIG["n_visual_tokens"],
        device=device,
        llm=llm,
        tokenizer=tokenizer,
        connector=connector,
        texts=texts  # Concatenated question + caption
    )

    return inputs_embeds, attention_mask, input_ids, image_token_idx

In [ ]:
# LOAD MODELS, LLM TOKENIZER, VISION PROCESSOR

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LLM_NAME = "Qwen/Qwen2.5-0.5B"
VISION_NAME = "google/siglip2-base-patch16-224"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME, dtype=torch.float32
).to(DEVICE)

vision_processor = AutoProcessor.from_pretrained(VISION_NAME, use_fast=True)
vision_model = AutoModel.from_pretrained(
    VISION_NAME, dtype=torch.float32
).to(DEVICE)

# ADD IMAGE TOKEN to LLM tokenizer vocabulary so it recognizes images
IMAGE_TOKEN = "<image>"
if IMAGE_TOKEN not in tokenizer.get_vocab():
    tokenizer.add_special_tokens({"additional_special_tokens": [IMAGE_TOKEN]})
    llm.resize_token_embeddings(len(tokenizer))

# CONSTRUCT NANO CHIMERA MODEL (QWEN 2.5 + SIGILP + CONNECTOR)
# SigLIP Embedding output dimension
vision_dim = vision_model.config.vision_config.hidden_size
# LLM Embedding output dimension
llm_dim = llm.config.hidden_size

connector = VisionConnector(
    vision_dim=vision_dim,
    llm_dim=llm_dim,
    hidden_dims=EX_CONFIG["connector_hidden_dims"],
    device=DEVICE
).to(DEVICE).float()
nano_chimera = NanoChimera(
    vision_encoder=vision_model.vision_model,
    llm=llm,
    connector=connector
).to(DEVICE)

NanoChimera

### 2.3 Discussion on simple training evaluation metrics

As we already know LLM evaluation is a non-trivial task due to all the nuances of language, then multimodal LLM (MLLM) being a strict superset of these models, we can easily see the complexity grows as the number of modalities grows, strictly. Real performance evaluation is usually done through benchmarks and human/llm-as-a-judge evaluation, although for training such expensive evaluation metrics cannot be used.


So besides **negative log-likelihood (NLL) loss** or **cross-entropy loss** for raw token classification, we would benefit from an interpretable metric to evaluate (similar to accuracy, recall or f1), which for LLMs pretraining its usually either of these 2:

1) Token-Accuracy: measures the fraction of correctly predicted tokens, ignoring masked tokens (e.g. image tokens or padding), in other words: **Did the model’s argmax token match the label?**. Its easy to interpret but satures quickly and ignores near-misses and confidence!


$
\text{Token Accuracy}
=
\frac{1}{|\mathcal{M}|}
\sum_{t \in \mathcal{M}}
\mathbf{1}\!\left[ \hat{y}_t = y_t \right]
$

2) Perplexity: the exponential of the average negative log-likelihood. In other words: **On average, how many equally likely tokens the model is confused between.** When the PPL = 1, then the prediction is perfect, otherwise PPL = 5 its a random guess between 5 tokens, the lower the better.

$
\text{Perplexity}
=
\exp\!\left(
\mathcal{L}_{\mathrm{NLL}}
\right)
=
\exp\!\left(
-\frac{1}{|\mathcal{M}|}
\sum_{t \in \mathcal{M}}
\log p_\theta\!\left( y_t \mid x_{<t} \right)
\right)
$

So in conclusion, Token Accuracy can be used for sanity check and debugging, but to get an actual intuition of performance north-star we will use perplexity.

In [ ]:
class LMStats:
    """Accumulates language modeling statistics."""
    def __init__(self):
        self.loss_sum = 0.0
        self.token_count = 0
        self.correct = 0

    def update(self, loss, logits, labels):
        """
        loss: scalar CE loss (already reduced)
        logits: [1, T, V]
        labels: [1, T] with -100 masked tokens
        """
        mask = labels != -100
        n_tokens = mask.sum().item()

        self.loss_sum += loss.item() * n_tokens
        self.token_count += n_tokens

        with torch.no_grad():
            preds = logits.argmax(dim=-1)
            self.correct += ((preds == labels) & mask).sum().item()

    def avg_loss(self):
        return self.loss_sum / max(self.token_count, 1)

    def perplexity(self):
        return math.exp(self.avg_loss())

    def accuracy(self):
        return self.correct / max(self.token_count, 1)

### 2.4 Training Pipeline definition

In [ ]:
def train_one_epoch(model,
                    loader,
                    optimizer,
                    scheduler,
                    connector,
                    criterion,
                    device,
                    EX_CONFIG,
                    warmup_steps,
                    global_step):
    K = EX_CONFIG["n_visual_tokens"]
    model.train()
    total_loss, total_ppl, total_acc, total_tokens = 0.0, 0.0, 0.0, 0
    pbar = tqdm(loader, desc="Training", unit="batch")
    
    for batch in pbar:
        imgs = batch["image"].to(device)
        captions = batch["caption"]

        # --------------------------------------------------------
        # 1. Use a neutral inference prompt (NO image info)
        # --------------------------------------------------------
        question_input = "<image> What is in the picture?"
        texts = [question_input] * len(captions)  # Adding the same question for every sample

        # --------------------------------------------------------
        # 2. Tokenize the text prompt along with the captions
        # --------------------------------------------------------
        input_ids = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).input_ids.to(device)
        attention_mask = input_ids.ne(tokenizer.pad_token_id).long()

        # --------------------------------------------------------
        # 3. Merge image embeddings with the question prompt
        # --------------------------------------------------------
        inputs_embeds, attention_mask, image_token_idx = merge_text_and_visual_embeddings(
            images=imgs,
            K=EX_CONFIG["n_visual_tokens"],
            device=device,
            llm=model.llm,
            tokenizer=tokenizer,
            connector=connector,
            texts=texts  # Using the question prompt
        )

        B, L, D = inputs_embeds.size()
        labels = input_ids.new_full((B, L), -100)  # -100 is ignored by cross-entropy

        # --------------------------------------------------------
        # 4. Set labels for the text following the <image> token
        # --------------------------------------------------------
        for b in range(B):
            idx = image_token_idx[b]
            labels[b, :idx] = input_ids[b, :idx]  # Copy tokens before <image>
            labels[b, idx + K: idx + K + (input_ids.size(1) - (idx + 1))] = input_ids[b, idx+1:]

        # --------------------------------------------------------
        # 5. Forward pass
        # --------------------------------------------------------
        outputs = model.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=None  # We'll handle labels manually
        )
        logits = outputs.logits  # [B, L, vocab_size]

        # --------------------------------------------------------
        # 6. Compute loss on shifted logits/labels
        # --------------------------------------------------------
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:]  # Standard causal LM shift

        raw_loss = criterion(
            shift_logits.reshape(-1, shift_logits.size(-1)),
            shift_labels.reshape(-1)
        )

        # --------------------------------------------------------
        # 7. Backward pass + optimizer step
        # --------------------------------------------------------
        (raw_loss / EX_CONFIG["grad_accum_steps"]).backward()
        if (global_step + 1) % EX_CONFIG["grad_accum_steps"] == 0:
            optimizer.step()
            optimizer.zero_grad()
            if scheduler:
                scheduler.step()

        # --------------------------------------------------------
        # 8. Compute metrics
        # --------------------------------------------------------
        total_loss += raw_loss.item() * B
        total_tokens += (shift_labels != -100).sum().item()

        preds = shift_logits.argmax(dim=-1)
        correct = (preds == shift_labels) & (shift_labels != -100)
        total_acc += correct.sum().item()

        # Calculate perplexity
        avg_loss = total_loss / total_tokens
        avg_ppl = torch.exp(torch.tensor(avg_loss)).item()
        avg_acc = total_acc / total_tokens

        # Update progress bar with accuracy, loss, and perplexity
        pbar.set_postfix({
            "acc": f"{avg_acc:.4f}",
            "loss": f"{avg_loss:.4f}",
            "ppl": f"{avg_ppl:.2f}",
            "tokens": total_tokens
        })

        global_step += 1

    avg_loss = total_loss / total_tokens
    avg_acc = total_acc / total_tokens
    avg_ppl = torch.exp(torch.tensor(avg_loss)).item()

    # Return the values that are expected by the calling function
    return avg_loss, avg_ppl, avg_acc, total_tokens, global_step

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device, EX_CONFIG):
    K = EX_CONFIG["n_visual_tokens"]
    model.eval()
    total_loss, total_ppl, total_acc, total_tokens = 0.0, 0.0, 0.0, 0
    pbar = tqdm(loader, desc="Evaluating", unit="batch")

    with torch.no_grad():  # Wir verhindern die Berechnung von Gradienten
        for batch in pbar:
            imgs = batch["image"].to(device)
            captions = batch["caption"]

            # 1. Tokenize text
            input_ids = tokenizer(captions, return_tensors="pt", padding=True, truncation=True).input_ids.to(device)
            attention_mask = input_ids.ne(tokenizer.pad_token_id).long()

            # 2. Encode images + merge embeddings
            inputs_embeds, attention_mask, image_token_idx = merge_text_and_visual_embeddings(
                images=imgs,
                K=EX_CONFIG["n_visual_tokens"],
                device=device,
                llm=model.llm,
                tokenizer=tokenizer,
                connector=connector,
                texts=captions
            )

            B, L, D = inputs_embeds.size()
            labels = input_ids.new_full((B, L), -100)  # -100 wird vom Cross-Entropy als ignoriert behandelt

            for b in range(B):
                idx = image_token_idx[b]
                # Kopiere Text-Token vor <image>
                labels[b, :idx] = input_ids[b, :idx]
                # Kopiere Text-Token nach <image>, verschoben um K visuelle Token
                labels[b, idx + K: idx + K + (input_ids.size(1) - (idx + 1))] = input_ids[b, idx+1:]

            # 4. Forward pass
            outputs = model.llm(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                labels=None  # Wir behandeln die Labels manuell
            )
            logits = outputs.logits  # [B, L, vocab_size]

            # 5. Verlust berechnen (verschobene Logits/Labels)
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:]  # Standard Causal LM Shift

            raw_loss = criterion(
                shift_logits.reshape(-1, shift_logits.size(-1)),
                shift_labels.reshape(-1)
            )

            # 6. Metriken
            total_loss += raw_loss.item() * B
            total_tokens += (shift_labels != -100).sum().item()

            # Optional: Token-Genauigkeit berechnen
            preds = shift_logits.argmax(dim=-1)
            correct = (preds == shift_labels) & (shift_labels != -100)
            total_acc += correct.sum().item()

            # Calculate perplexity
            avg_loss = total_loss / total_tokens
            avg_ppl = torch.exp(torch.tensor(avg_loss)).item()
            avg_acc = total_acc / total_tokens

            pbar.set_postfix({
                "acc": f"{avg_acc:.4f}",
                "loss": f"{avg_loss:.4f}",
                "ppl": f"{avg_ppl:.2f}",
                "tokens": total_tokens
            })

    # Berechne durchschnittliche Metriken
    avg_loss = total_loss / total_tokens
    avg_ppl = torch.exp(torch.tensor(avg_loss)).item()
    avg_acc = total_acc / total_tokens

    return avg_loss, avg_ppl, avg_acc, total_tokens

In [ ]:
def model_training(
    n_epochs,
    nano_chimera,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    model_name="best_nano_chimera.pt"
):
    """
    Standard training loop for multimodal LLMs.
    - Tracks loss, perplexity, token accuracy, token counts
    - Saves best checkpoint
    - Saves JSON + PNG + PDF report for each run
    """

    # 0. Experiment naming & folders
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    dataset_size = EX_CONFIG["dataset_size"]
    batch_size = EX_CONFIG["batch_size"]
    hidden = "x".join(map(str, EX_CONFIG["connector_hidden_dims"]))
    run_name = f"run_{timestamp}_ds{dataset_size}_bs{batch_size}_h{hidden}_ep{n_epochs}"

    output_dir = os.path.join("runs", run_name)
    os.makedirs(output_dir, exist_ok=True)

    best_model_path = os.path.join(output_dir, model_name)
    json_path = os.path.join(output_dir, "results.json")
    plot_path = os.path.join(output_dir, "training_metrics.pdf")
    png_path = os.path.join(output_dir, "training_metrics.png")
    pdf_path = os.path.join(output_dir, "report.pdf")

    # 1. Training bookkeeping
    best_val_loss = float("inf")

    train_losses, train_accs, train_ppls = [], [], []
    val_losses, val_accs, val_ppls = [], [], []
    train_token_counts, val_token_counts = [], []

    global_step = 0
    times = None

    # 2. Training loop
    for epoch in range(n_epochs):
        start = time.time()
        
        train_loss, train_ppl, train_acc, train_tokens, global_step = train_one_epoch(
            nano_chimera,
            train_loader,
            optimizer,
            scheduler,
            connector,
            criterion,
            device,
            EX_CONFIG,
            warmup_steps,
            global_step
        )

        val_loss, val_ppl, val_acc, val_tokens = evaluate(
            nano_chimera,
            val_loader,
            criterion,
            device,
            EX_CONFIG
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(nano_chimera.state_dict(), best_model_path)

        logger.info(
            f"Epoch {epoch+1}/{n_epochs} | "
            f"Train Loss {train_loss:.4f} | PPL {train_ppl:.2f} | Acc {train_acc:.4f} | Tokens {train_tokens} | "
            f"Val Loss {val_loss:.4f} | PPL {val_ppl:.2f} | Acc {val_acc:.4f} | Tokens {val_tokens} | "
            f"Time {time.time()-start:.1f}s"
        )

        train_losses.append(train_loss)
        train_accs.append(train_acc)
        train_ppls.append(train_ppl)
        train_token_counts.append(train_tokens)

        val_losses.append(val_loss)
        val_accs.append(val_acc)
        val_ppls.append(val_ppl)
        val_token_counts.append(val_tokens)

    # 3. Plot metrics (Token count, accuracy, perplexity, loss...)
    train_token_cum = np.cumsum(train_token_counts)
    val_token_cum = np.cumsum(val_token_counts)

    fig = plt.figure(figsize=(20,5))

    # Prepare data and titles
    plots = [
        (train_losses, val_losses, "Loss"),
        (train_accs, val_accs, "Token Accuracy"),
        (train_token_cum, val_token_cum, "Cumulative Tokens"),
        (train_ppls, val_ppls, "Perplexity")
    ]
    
    plt.figure(figsize=(20, 5))  # adjust figure size
    
    for i, (train_data, val_data, title) in enumerate(plots, 1):
        plt.subplot(1, 4, i)
        plt.plot(train_data, label="Train")
        plt.plot(val_data, label="Val")
        plt.title(title)
        plt.grid(True)
        plt.legend()
    
    plt.tight_layout()
    plt.savefig(plot_path, dpi=150)
    plt.savefig(png_path, dpi=150)

    plt.close()

    # 4. Save RUN Report JSON
    run_summary = {
        "run_name": run_name,
        "timestamp": timestamp,
        "config": EX_CONFIG,
        "metrics": {
            "train_loss": train_losses,
            "val_loss": val_losses,
            "train_accuracy": train_accs,
            "val_accuracy": val_accs,
            "train_perplexity": train_ppls,
            "val_perplexity": val_ppls,
            "train_tokens_epoch": train_token_counts,
            "val_tokens_epoch": val_token_counts,
            "train_tokens_cumulative": train_token_cum.tolist(),
            "val_tokens_cumulative": val_token_cum.tolist(),
        },
        "artifacts": {
            "plot": plot_path,
            "best_model": best_model_path,
            "pdf_report": pdf_path
        },
        "best_val_loss": best_val_loss
    }

    with open(json_path, "w") as f:
        json.dump(run_summary, f, indent=2)

    # 5. Generate PDF report
    with PdfPages(pdf_path) as pdf:
        fig = plt.figure(figsize=(11,8))


        plt.text(0.01, 0.95, "Final Metrics Summary", fontsize=14)

        summary_lines = [
            f"Final Train Loss: {train_losses[-1]:.4f}",
            f"Final Val Loss:   {val_losses[-1]:.4f}",
            f"Final Train Acc:  {train_accs[-1]:.4f}",
            f"Final Val Acc:    {val_accs[-1]:.4f}",
            f"Final Train PPL:  {train_ppls[-1]:.2f}",
            f"Final Val PPL:    {val_ppls[-1]:.2f}",
            "",
            f"Total Train Tokens: {int(train_token_cum[-1])}",
            f"Total Val Tokens:   {int(val_token_cum[-1])}",
            "",
            f"Best Validation Loss: {best_val_loss:.4f}"
        ]

        y = 0.88
        for line in summary_lines:
            plt.text(0.01, y, line, fontsize=11)
            y -= 0.05

        y = 0.85
        for k, v in EX_CONFIG.items():
            plt.text(0.01, y, f"{k}: {v}", fontsize=9)
            y -= 0.03

        plt.axis("off")
        pdf.savefig(fig)
        plt.close()

        img = plt.imread(png_path)
        fig = plt.figure(figsize=(11,5))
        plt.imshow(img)
        plt.axis("off")
        pdf.savefig(fig)
        plt.close()

    logger.info(f"Run saved to {output_dir}")

    return (
        train_losses,
        train_accs,
        val_losses,
        val_accs,
        train_token_counts,
        val_token_counts
    )

### 2.5 Example (Train + Inference)


In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=-100,
    label_smoothing=EX_CONFIG["label_smoothing"])
criterion = criterion.to(DEVICE)

optimizer = torch.optim.AdamW(
    connector.parameters(),
    lr=EX_CONFIG["learning_rate"],
    weight_decay=EX_CONFIG["weight_decay"]
)

total_steps = len(data) // EX_CONFIG["grad_accum_steps"]
warmup_steps = int(EX_CONFIG["warmup_ratio"] * total_steps)

scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=EX_CONFIG["scheduler_start_factor"],
    total_iters=warmup_steps
)

connector.train()
optimizer.zero_grad()

train_losses, train_accs, val_losses, val_accs, train_token_counts, val_token_counts = model_training(
    n_epochs=EX_CONFIG["epochs"],
    nano_chimera=nano_chimera,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=DEVICE,
    model_name="best_nano_chimera.pt"
)

#### Inference!

This is a big TODO, i'm not sure this is working properly

In [ ]:
@torch.no_grad()
def inference_nano_chimera(
    nano_chimera,
    dataloader,
    tokenizer,
    connector,
    K,
    device,
    max_new_tokens=None,
    min_new_tokens=None,
    do_sample=None,
    temperature=None,
    top_p=None,
    repetition_penalty=None,
    no_repeat_ngram_size=None,
    text_prompt=None  # New parameter for custom text string
):
    nano_chimera.eval()
    results = []

    # Grab standard defaults from the model config
    config = nano_chimera.llm.config
    max_new_tokens = max_new_tokens if max_new_tokens is not None else getattr(config, "max_new_tokens", 20)
    min_new_tokens = min_new_tokens if min_new_tokens is not None else getattr(config, "min_new_tokens", 0)
    do_sample = do_sample if do_sample is not None else getattr(config, "do_sample", False)
    temperature = temperature if temperature is not None else getattr(config, "temperature", 1.0)
    top_p = top_p if top_p is not None else getattr(config, "top_p", 1.0)
    repetition_penalty = repetition_penalty if repetition_penalty is not None else getattr(config, "repetition_penalty", 1.0)
    no_repeat_ngram_size = no_repeat_ngram_size if no_repeat_ngram_size is not None else getattr(config, "no_repeat_ngram_size", 0)

    # Set the default text prompt or use the custom text prompt provided
    if text_prompt is None:
        text_prompt = "<image> Please describe the contents of this image in one sentence."
    else:
        text_prompt = f"<image> {text_prompt}"

    for batch in dataloader:
        images = batch["image"]
        captions = batch["caption"]  # for logging only

        # Merge text + visual embeddings using the provided text prompt
        inputs_embeds, attention_mask, image_token_idx = merge_text_and_visual_embeddings(
            images=images,
            K=K,
            device=device,
            llm=nano_chimera.llm,
            tokenizer=tokenizer,
            connector=connector,
            texts=[text_prompt] * len(images)  # Ensure the prompt is applied to all images in the batch
        )

        # Generate
        generated_ids = nano_chimera.llm.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            no_repeat_ngram_size=no_repeat_ngram_size,
        )

        # Decode predictions
        for i in range(len(images)):
            pred_caption = tokenizer.decode(generated_ids[i], skip_special_tokens=True)
            results.append({
                "image": images[i],
                "caption": captions[i],
                "generated": pred_caption
            })

    return results

In [ ]:
# Call with a proposed set of parameters for inference
results = inference_nano_chimera(
    nano_chimera=nano_chimera,
    dataloader=test_loader,
    tokenizer=tokenizer,
    connector=connector,
    K=EX_CONFIG["n_visual_tokens"],
    device=DEVICE,
    max_new_tokens=50,           # Max number of tokens to generate
    min_new_tokens=10,           # Minimum number of tokens to generate
    do_sample=True,              # Enable random sampling
    temperature=0.7,             # Randomness in generation (0.7 for balance)
    top_p=0.9,                   # Nucleus sampling (top-p)
    repetition_penalty=1.2,      # Penalize repetition of n-grams
    no_repeat_ngram_size=2,       # Prevent repeating 2-grams
    text_prompt="Describe this image in one sentence."
)

In [ ]:
# Print first 5 results
for r in results[20:30]:
    print("GT:", r["caption"])  # Ground truth caption
    print("GEN:", r["generated"])  # Generated caption
    img = r["image"].permute(1, 2, 0).cpu().numpy()
    plt.imshow(img)  # Display the image
    plt.axis('off')  # Hide axis
    plt.show()

### 2.6 Pipeline Extension

In typical VLM training scenarios, analogously to LLM training, there are multiple phases not only a single iteration loop. For LLMs this is typically decomposed in the stages:

1. Pretraining: Similar to our training, the model learns to predict next tokens from **unsupervised corpora**, learning representations of language along the way.

2. Supervised Fine-Tuning: Through a **golden standard supervised Q&A dataset** the model learns to be a helpful assistant and to follow instructions.

3. Preference alingment (Reinforcement Learning from Human Feedback): Further refinement with Q&A style data with RL based algorithms like PPO, DPO, GRPO...

> However in multimodal VLM's the pipeline usually goes as following:

1. **Stage 1: Vision-Language Feature Alignment (Pre-training)**: In this phase the objective is align visual features from Visual Encoders with Language Model embeddings. Typical configurations of hyperparameters are:

    1-2 epochs on ~600K image-text pairs Batch Size: 256-512, Learning Rate: 1e-3 to 2e-3 (Bigger)

2. **Stage 2: Visual Instruction Tuning (SFT):**
  In this phase the objective is to teach the model to follow multimodal instructions. Typical configurations of hyperparameters are:

    3-5 epochs on ~150K instruction-examples Batch Size: 128-256 Learning Rate: 2e-5 (much smaller).




So if we want to refine the quality of our model to go beyond a simple pretraining modality alignment, we need to perform yet another training using visual instruction tuning datasets. Luckily for us again, huggingface contains such already processed datasets. For example: liuhaotian/LLaVA-Instruct-150K.


In [ ]:
# This is a sample
VSFT_CONFIG = {
  # Data
  "train_size": 16,
  "test/val_ratio": 0.5,
  "batch_size": 16,

  # Optimization algorithm
  "learning_rate": 2e-5,
  "weight_decay": 0.01,

  # Connector parameters
  "n_visual_tokens": 32,
  "connector_hidden_dims": (4096, ),

  # training
  "epochs": 3,
  "grad_accum_steps": 8,
  "warmup_ratio": 0.05,
  "max_grad_norm": 1.0,
  "scheduler_start_factor": 0.1,

  # logging
  "log_every": 128,
  "eval_every": 256,

}

In [ ]:
# LOAD DATA
# Create a temporary custom Dataset class to hold the processed data
class LLaVASFTDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data = data_list
        self.transform = transform # Keep transform for potential future image augmentations

    def __getitem__(self, idx):
        item = self.data[idx]
        image_data = item["image"]
        caption = item["caption"]

        # Explicitly load image if it's a string filename
        if isinstance(image_data, str):
            try:
                # Assuming the image file is accessible via this path
                image = Image.open(image_data).convert("RGB")
            except FileNotFoundError as e:
                logger.error(f"Image file not found at path: {image_data}. Skipping sample. Error: {e}")
                # Return a dummy image or raise an error to indicate problem
                raise ValueError(f"Image file not found: {image_data}") from e
            except Exception as e:
                logger.error(f"Failed to open image {image_data}. Skipping sample. Error: {e}")
                raise ValueError(f"Failed to open image {image_data}") from e
        else:
            # If it's already a PIL Image (or other image object type), use it directly
            image = image_data

        if self.transform is not None:
            image = self.transform(image) # Assuming transform applies to image

        return {
            "image": image,
            "caption": caption
        }

    def __len__(self):
        return len(self.data)

# Dataset name for SFT
LLAVA_SFT_DATASET_NAME = "liuhaotian/LLaVA-Instruct-150K"

# Manually load and process the LLaVA-Instruct-150K dataset to extract images and GPT responses
# This bypasses the VLM_Dataset's expectation of a direct 'answer' key.
print(f"Loading and processing {LLAVA_SFT_DATASET_NAME} for SFT...")
processed_sft_data = []
dataset_stream_sft = load_dataset(
    LLAVA_SFT_DATASET_NAME,
    split="train", # Assuming we take the 'train' split of the SFT dataset
    streaming=True
).shuffle(seed=SEED, buffer_size=1000) # Buffer size for streaming shuffle

count = 0
for row in dataset_stream_sft:
    if count >= VSFT_CONFIG["train_size"]:
        break
    image = row["image"]

    # Extract the full conversation as the caption
    current_caption_parts = []
    has_image_instruction = False

    for conv in row["conversations"]:
        if conv["from"] == "human":
            # Check if the human instruction includes an image token
            if "<image>" in conv["value"]:
                has_image_instruction = True
            # Remove the <image> token from the human's input for the text part
            question_text = conv["value"].replace("<image>", "").strip()
            if question_text:
                current_caption_parts.append(f"User: {question_text}")
        elif conv["from"] == "gpt":
            if conv["value"]:
                current_caption_parts.append(f"Assistant: {conv["value"]}")

    # Combine into a single string for the caption
    final_caption = " ".join(current_caption_parts).strip()

    # Ensure image, a non-empty conversation, and explicit image instruction are present
    if image and final_caption and has_image_instruction:
        processed_sft_data.append({
            "image": image,
            "caption": final_caption
        })
        count += 1
print(f"Finished processing {len(processed_sft_data)} samples.")



# Instantiate the custom dataset
full_sft_dataset = LLaVASFTDataset(processed_sft_data)

# Split the full SFT dataset into training, validation, and test sets
# Adjust ratios based on typical practice for SFT, e.g., 80% train, 10% val, 10% test
sft_train_ratio = 1 - VSFT_CONFIG["test/val_ratio"]
sft_val_ratio = VSFT_CONFIG["test/val_ratio"] / 2
sft_test_ratio = VSFT_CONFIG["test/val_ratio"] / 2

# n_total_sft is the total number of samples loaded for SFT
n_total_sft = len(full_sft_dataset)
n_sft_train = int(sft_train_ratio * n_total_sft)
n_sft_val = int(sft_val_ratio * n_total_sft)
n_sft_test = n_total_sft - n_sft_train - n_sft_val # Ensure all samples are used

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset, test_dataset = random_split(
    full_sft_dataset,
    [n_sft_train, n_sft_val, n_sft_test],
    generator=generator
)

# Now, create the DataLoaders with the collate_fn
train_loader = DataLoader(
    train_dataset, batch_size=VSFT_CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset, batch_size=VSFT_CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
)
test_loader = DataLoader(
    test_dataset, batch_size=VSFT_CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
)

print("SFT DataLoaders created successfully.")

In [ ]:
# SFT Training Pipeline
# TODO: Search a dataset that is real, LLAVA paper one lacks images, images are strings of filenames rather than actual images LOL
"""
criterion_sft = nn.CrossEntropyLoss()
criterion_sft = criterion_sft.to(DEVICE)

optimizer_sft = torch.optim.AdamW(
    connector.parameters(),
    lr=VSFT_CONFIG["learning_rate"],
    weight_decay=VSFT_CONFIG["weight_decay"]
)

total_steps_sft = len(full_sft_dataset) // VSFT_CONFIG["grad_accum_steps"]
warmup_steps_sft = int(VSFT_CONFIG["warmup_ratio"] * total_steps_sft)

scheduler_sft = torch.optim.lr_scheduler.LinearLR(
    optimizer_sft,
    start_factor=VSFT_CONFIG["scheduler_start_factor"],
    total_iters=warmup_steps_sft
)

connector.train()
optimizer_sft.zero_grad()


train_losses_sft, train_accs_sft, val_losses_sft, val_accs_sft, train_token_counts_sft, val_token_counts_sft = model_training(
    n_epochs=VSFT_CONFIG["epochs"],
    nano_chimera=nano_chimera,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_sft,
    criterion=criterion_sft,
    device=DEVICE,
    model_name="best_nano_chimera_sft.pt" # Save a new model for SFT
)
"""
None

In [ ]:
full_sft_dataset.data[0]

In [ ]:
# TRAINING PIPELINE
# This only needs a small tweak in the inputs of the training loop :)

## 3. Experiments

In this section we are going to execute some experiments to try to optimize this NanoChimera architecture as much as possible with different configurations of hyperparameters and even small architecture size changes. Given this multi-variable optimization landscape, we need to take into account all of the hyperparameters and factors that can affect the model performance.

### 3.1 Basic Adapter (1 layer, 4096 hidden units)

In [ ]:
config = {
    "DATASET_SIZE": 10000,
    "BATCH_SIZE": 16,
    "NUM_EPOCHS": 20,
    "LEARNING_RATE": 1e-5
}

### 3.2 Two Layer Adapter



In [ ]:
config = {
    "DATASET_SIZE": 10000,
    "BATCH_SIZE": 16,
    "NUM_EPOCHS": 20,
    "LEARNING_RATE": 1e-5
}

## 4. Evaluation pipeline


In this section, we evaluate the performance of our multimodal model using **task-specific benchmarks** rather than relying solely on generic metrics like loss or perplexity.  

While metrics such as training/validation loss or perplexity are useful for monitoring model convergence, they **do not fully capture the model's reasoning abilities or real-world performance** on multimodal tasks. For example, a model might achieve low loss but still fail to answer visual questions correctly or reason about images in context.  

To address this, we focus on the following benchmarks:  

- **MMBench**: Tests general multimodal reasoning, commonsense understanding, and scene comprehension.  
- **VQAv2**: Standard benchmark for visual question answering, assessing object recognition, counting, and attribute understanding.  

By evaluating on these benchmarks, we obtain a **more accurate and practical measure of the model's capabilities**, aligning our results with published baselines such as LLaVA.


Although there's no silver bullet benchmark, most of them are curated and offer high quality assessing capabilities of AI systems. **lm_eval_harness** has become the standard de-facto evaluation system for Language models (LM) , similarly this framework **lmms_eval** is becoming also the de-facto for multimodal language models (MLM)

In [ ]:
# GENERAL LLMS EVAL HELP
#!lmms-eval --help
# THIS SERVES TO LIST THE EVALUATION TASKS AVAILABLE (MULTIMODAL TASKS)
#!python -m lmms_eval --tasks list

In [ ]:
class NanoChimeraLMMSWrapper(lmms):
    """
    LMMS eval model wrapper for NanoChimera-style models.
    Implements minimal LMMS expected methods in a robust, notebook-friendly way.
    - Single-example and small-batch compatible
    - Careful device/dtype handling
    - Defensive checks and clear error messages
    """
    def __init__(
        self,
        nano_chimera,
        tokenizer,
        vision_processor,
        vision_model,
        connector,
        n_visual_tokens: int,
        device: Optional[torch.device] = None
    ):
        super().__init__()
        self.nano_chimera = nano_chimera
        self.tokenizer = tokenizer
        self.vision_processor = vision_processor
        # vision_model may wrap; prefer the model device
        self.vision_model = getattr(vision_model, "vision_model", vision_model)
        self.connector = connector
        self.n_visual_tokens = int(n_visual_tokens)
        self.device = device or getattr(nano_chimera.llm, "device", torch.device("cpu"))

        # LMMS-eval expected attributes (keep conservative defaults)
        self.is_vllm = False
        self.max_length = 2048
        self.max_gen_toks = 32

    ########################################
    # High-level LMMS-compatible entrypoints
    ########################################

    def generate_until(self, requests: list[Instance]) -> list[str]:
        outputs = []

        for inst in requests:
            # Legacy/simple model unpack
            question = inst.args[0]
            gen_kwargs = inst.args[1].copy()  # copy to avoid modifying original
            doc_to_visual = inst.args[2]

            # Extract image if present
            image = None
            if inst.doc is not None:
                visuals = doc_to_visual(inst.doc)
                image = visuals[0] if visuals else None

            # Remove 'max_new_tokens' from gen_kwargs to avoid duplicate argument
            max_new_tokens = gen_kwargs.pop("max_new_tokens", self.max_gen_toks)

            # Generate answer
            answer = self.generate_answer(
                image=image,
                prompt=question,
                max_new_tokens=max_new_tokens,
                **{k: v for k, v in gen_kwargs.items() if k != "until"}
            )

            outputs.append(answer)

        return outputs


    def generate_until_multi_round(self, requests: List[Any]) -> List[List[Dict[str, str]]]:
        """
        Multi-round (conversation) support: return list of lists of responses.
        Keep a minimal, LMMS-friendly contract.
        """
        all_rounds = []
        for req in requests:
            # assume each req is same shape as single-round for our VQA use-case
            out = self.generate_until([req])
            all_rounds.append(out)
        return all_rounds

    def loglikelihood(self, requests: List[Instance]) -> List[tuple[float, bool]]:
        return [(0.0, False) for _ in requests]  # LMMS won’t crash (Placeholder)


    def generate_answer(self, image, prompt: str, max_new_tokens: int = 20, **kwargs) -> str:
        """
        Generate an answer conditioned on a single image + prompt.
        - image: PIL.Image or list[...]. We support single-image inputs.
        - prompt: textual prompt (string). We will inject IMAGE_TOKEN where needed.
        Returns: generated answer string (prompt stripped when possible).
        """
        # ensure evaluation mode
        self.nano_chimera.eval()

        #  prepare visual features
        if image is None:
            # Small-sample / debug fallback
            return "[no image provided; placeholder answer]"

        # vision processing -> tensor on vision model device
        inputs = self.vision_processor(images=image, return_tensors="pt")
        vision_device = next(self.vision_model.parameters()).device
        inputs = {k: v.to(vision_device) for k, v in inputs.items()}

        with torch.no_grad():
            vis_out = self.vision_model(**inputs)
            # last_hidden_state shape: (B, L, D). We take first n_visual_tokens
            vis_feats = getattr(vis_out, "last_hidden_state", vis_out)[..., :self.n_visual_tokens, :]

        # move connector to vision embeddings device/dtype if possible
        connector_device = getattr(self.connector, "device", vision_device)
        vis_feats = vis_feats.to(connector_device)
        with torch.no_grad():
            vis_embeds = self.connector(vis_feats)  # expected [B, K, llm_dim] or similar

        #  prepare text embeddings
        # Insert/expect IMAGE_TOKEN to mark where visual tokens go
        text_input = f"{IMAGE_TOKEN} {prompt}"
        tokenized = self.tokenizer(text_input, return_tensors="pt")
        input_ids = tokenized.input_ids.to(self.device)

        # get token id for IMAGE_TOKEN and find its index (robust)
        image_token_id = self.tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)
        if image_token_id is None:
            # try special tokens
            image_token_id = tokenized["input_ids"][0][(tokenized["input_ids"][0] == image_token_id)].tolist() if False else None

        # input embeddings from LLM
        embed_layer = self.nano_chimera.llm.get_input_embeddings()
        text_embeds = embed_layer(input_ids).to(vis_embeds.dtype)

        # find image token index in the input ids
        idx_positions = (input_ids == image_token_id).nonzero(as_tuple=False)
        if idx_positions.shape[0] == 0:
            # fallback: try to find a textual marker or raise clear error
            raise ValueError(
                f"IMAGE_TOKEN ({IMAGE_TOKEN}) not found in tokenized input. "
                "Ensure IMAGE_TOKEN matches your tokenizer special token exactly."
            )
        # take the first occurrence in first batch
        idx = idx_positions[0, 1].item()

        #  build final inputs_embeds by concatenation
        # handle batch dim (we assume single example B=1 here)
        # split text embeddings around the IMAGE_TOKEN position
        left = text_embeds[:, :idx, :]
        right = text_embeds[:, idx + 1 :, :]

        # vis_embeds shape must be [1, K, D] and dtype matches text_embeds
        if vis_embeds.ndim == 2:
            # sometimes connector returns flattened shape; try to reshape to (1, K, D)
            vis_embeds = vis_embeds.unsqueeze(0)
        vis_embeds = vis_embeds.to(text_embeds.dtype)
        if left.size(0) != 1:
            # ensure batch dimension is 1 (LMMS small-sample mode)
            pass

        input_embeds = torch.cat([left, vis_embeds, right], dim=1).to(self.device)
        attention_mask = torch.ones(input_embeds.size()[:-1], device=self.device, dtype=torch.long)

        #  generation arguments (deterministic by default for eval)
        generation_kwargs = {
            "inputs_embeds": input_embeds,
            "attention_mask": attention_mask,
            "max_new_tokens": max_new_tokens,
            "pad_token_id": getattr(self.tokenizer, "pad_token_id", self.tokenizer.eos_token_id),
            "do_sample": False,
            "temperature": 0.0,
            **kwargs,
        }

        # ensure LLM device alignment
        try:
            # move embeds to LLM device if different
            llm_device = getattr(self.nano_chimera.llm, "device", self.device)
            input_embeds = input_embeds.to(llm_device)
            generation_kwargs["inputs_embeds"] = input_embeds
            generation_kwargs["attention_mask"] = generation_kwargs["attention_mask"].to(llm_device)
        except Exception:
            # best-effort only
            pass

        # - perform generation safely -
        with torch.no_grad():
            generated_ids = self.nano_chimera.llm.generate(**generation_kwargs)

        # decode the model output
        # if generate returns tensor of ids, decode; some LLMs return dicts
        if isinstance(generated_ids, torch.Tensor):
            gen_text = self.tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        else:
            # try to handle other return formats (e.g., list[str])
            gen_text = str(generated_ids)

        # strip the prompt if present (robustly)
        # try to remove the first instance of the input prompt (text_input)
        if text_input.strip() and text_input.strip() in gen_text:
            # remove only the first occurrence to keep any valid generated continuation
            gen_text = gen_text.replace(text_input.strip(), "", 1).strip()

        return gen_text.strip()
# -
# Small-sample evaluation config (two benchmarks, two samples each)
# -
lmms_model_wrapper = NanoChimeraLMMSWrapper(
    nano_chimera=nano_chimera,
    tokenizer=tokenizer,
    vision_processor=vision_processor,
    vision_model=vision_model,
    connector=connector,
    n_visual_tokens=EX_CONFIG["n_visual_tokens"],
    device=DEVICE
)

lmms_eval_config = {
    #"task_names": ["mmbench", "vqav2"],
    "task_names": ["llava_in_the_wild"],
    #"task_names": ["llava_bench_coco", "llava_in_the_wild", "llava_wilder_small"],
    "num_fewshot": 0,
    "batch_size": 1,
    "limit": 4,
    "verbose": True,
    "predict_only": True,
}


tm = TaskManager()
tasks = tm.load_task_or_group(
    ["llava_in_the_wild"],
    task_type="simple"
)
tasks["llava_in_the_wild"].config.full_docs = True

In [ ]:
lmms_evaluation_results = simple_evaluate(
    batch_size=lmms_eval_config["batch_size"],
    numpy_random_seed=SEED,
    random_seed=SEED,
    torch_random_seed=SEED,
    fewshot_random_seed=SEED,
    model=lmms_model_wrapper,
    tasks=lmms_eval_config["task_names"],
    num_fewshot=lmms_eval_config["num_fewshot"],
    limit=lmms_eval_config["limit"],
    predict_only=lmms_eval_config["predict_only"],
    model_args={
        "hf_token": os.environ.get("HUGGINGFACE_TOKEN")
    },
)

# print/save results neatly
print("LMMS Evaluation Results (small sample):")
print(lmms_evaluation_results)

In [ ]:
tm = TaskManager("INFO")

task_dict = get_task_dict(
    ["llava_in_the_wild"],
    tm,
    task_type="simple"
)

# Grab ONE task
task = list(task_dict.values())[0]

# Build a SINGLE request
task.build_all_requests(limit=5)

inst = task.instances[3]
print(inst)

print("TYPE:", type(inst))
print("IS Instance:", isinstance(inst, Instance))
print("ARGS:", inst.args)
print("HAS DOC", inst.doc is not None)
print("DOC KEYS:", inst.doc.keys())
print("REQUEST TYPE:", inst.request_type)


In [ ]:
# -
# Tiny dummy datasets for quick debugging
# -
# Replace these with small PIL images or generated tensors for testing
dummy_image = Image.new("RGB", (224, 224), color="white")
dummy_image2 = Image.new("RGB", (224, 224), color="gray")

# MMBench dummy
mmbench_dummy = [
    {"question": "What is the object in the scene?", "answer": "cat", "image": dummy_image},
    {"question": "What color is the car?", "answer": "red", "image": dummy_image2},
]

# VQAv2 dummy
vqav2_dummy = [
    {"question": "How many cats?", "answer": "2", "image": dummy_image},
    {"question": "What color is the sky?", "answer": "blue", "image": dummy_image2},
]

# -
# Prepare model wrapper
# -
lmms_model_wrapper = NanoChimeraLMMSWrapper(
    nano_chimera=nano_chimera,
    tokenizer=tokenizer,
    vision_processor=vision_processor,
    vision_model=vision_model,
    connector=connector,
    n_visual_tokens=EX_CONFIG["n_visual_tokens"],
    device=DEVICE
)

# -
# Evaluation config
# -
lmms_eval_config = {
    "task_names": ["MMBench", "VQAv2"],  # both benchmarks
    "num_fewshot": 0,
    "limit": 2,  # two examples per task for quick checks
    "verbose": True,
    "save_path": "./lmms_results.json"
}

# -
# Small-sample evaluation using dummy datasets
# -
lmms_evaluation_results = {}

# Manually run each benchmark with dummy data
for task_name, dummy_dataset in zip(lmms_eval_config["task_names"], [mmbench_dummy, vqav2_dummy]):
    print(f"\n=== Evaluating {task_name} ===")
    task_results = []
    for i, example in enumerate(dummy_dataset):
        output = lmms_model_wrapper.generate_answer(
            image=example["image"],
            prompt=example["question"]
        )
        result = {
            "question": example["question"],
            "ground_truth": example["answer"],
            "generated": output
        }
        task_results.append(result)
        if lmms_eval_config["verbose"]:
            print(f"[{i+1}/{len(dummy_dataset)}] Prompt: {example['question']}")
            print(f"Generated: {output}")
    lmms_evaluation_results[task_name] = task_results

# -
# Save results to JSON
# -
save_path = lmms_eval_config["save_path"]
with open(save_path, "w") as f:
    json.dump(lmms_evaluation_results, f, indent=2)

print(f"\n✅ Evaluation saved to {save_path}")
print("LMMS Evaluation Results (small sample):")
print(json.dumps(lmms_evaluation_results, indent=2))


In [ ]:
ll_results = lmms_model_wrapper.loglikelihood([
    {"prompt": "How many cats?", "image": dummy_image, "label": "2"}
])
print(ll_results)

## 5. Results

In this section we are going to discuss the different experiment results, as well as the winning model pipeline combination.

Also we are going to upload it to hugging-face to opensource it for everybody to use freely, although it lacks some refinement.



In [ ]:
# 1. PRESENT SINGLE-RESULTS TABLE
# Flatten results into a single DF
all_rows = []
for task, examples in lmms_evaluation_results.items():
    for ex in examples:
        all_rows.append({
            "Task": task,
            "Question": ex["question"],
            "Ground Truth": ex["ground_truth"],
            "Generated": ex["generated"]
        })

df = pd.DataFrame(all_rows)
df



In [ ]:
# 2, PRESENT AGGREGATED RESULTS TABLE
summary = {}
for task, examples in lmms_evaluation_results.items():
    correct = sum(ex["ground_truth"].lower() == ex["generated"].lower() for ex in examples)
    total = len(examples)
    summary[task] = {"accuracy": correct / total}

summary_df = pd.DataFrame(summary).T
summary_df


In [ ]:
# 3. PUSH THE MODEL TO HUGGINGFACE WITH A DESCRIPTION OF THE AGGREGATED RESULTS

model = None
tokenizer = None

repo_name = "NanoChimeraVLM-V1"
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

 ## 6. Conclusions